# Generación del dataset mock — Predicción de Open Rate

**Actividad Integradora 1 · Módulo 1: Fundamentos de MLOps**
**Rol 2 — Desarrollo de datos y pipeline**

Este notebook genera el dataset mock para el problema de predicción de apertura de notificaciones push (*Open Rate*), siguiendo los lineamientos del **punto 9** de la actividad.

El dataset **no** es ruido aleatorio: la variable objetivo `target_opened` se construye a partir de un modelo logístico latente que combina las variables de forma realista (un `historical_open_rate` alto sube la probabilidad de apertura, muchos días sin abrir la bajan, hay fatiga por exceso de pushes, etc.). Esto garantiza que los modelos de clasificación encuentren señal real.

> **Nota de flujo MLOps:** la versión productiva de este generador vive en `src/generate_data.py` y se ejecuta como etapa del pipeline DVC (`dvc repro`). Este notebook documenta y explora el proceso; el archivo CSV resultante se versiona con **DVC**, no con Git.


## 1. Parámetros de generación

Semilla fija para reproducibilidad total: con los mismos parámetros, el dataset es idéntico en cada ejecución.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# Parámetros (coherentes con params.yaml del pipeline)
N_ROWS = 35_000      # eventos de notificación
N_USERS = 8_000      # usuarios únicos
SEED = 42            # semilla de reproducibilidad
OUTPUT = Path("../data/raw/openrate.csv")

rng = np.random.default_rng(SEED)
print(f"Generando {N_ROWS:,} eventos para {N_USERS:,} usuarios (seed={SEED})")

Generando 35,000 eventos para 8,000 usuarios (seed=42)


## 2. Catálogos de categorías y efectos

Las distribuciones no son uniformes (más realista) y cada categoría tiene un **efecto** sobre el logit de apertura. Valores positivos → más probable que el usuario abra.


In [2]:
# Catálogos con distribución no uniforme
SITES = ["news", "ecommerce", "sports", "finance", "entertainment"]
SITE_P = [0.28, 0.26, 0.18, 0.12, 0.16]

CAMPAIGN_TYPES = ["promotional", "transactional", "reactivation", "newsletter"]
CAMPAIGN_P = [0.42, 0.23, 0.15, 0.20]

DEVICE_OS = ["android", "ios", "web"]
DEVICE_P = [0.55, 0.38, 0.07]

SEGMENTS = ["new", "engaged", "casual", "dormant"]
SEGMENT_P = [0.18, 0.30, 0.34, 0.18]

# Efectos sobre el logit de apertura
CAMPAIGN_EFFECT = {
    "promotional": -0.15,   # mucha promo, fácil de ignorar
    "transactional": 0.65,  # avisos relevantes -> se abren más
    "reactivation": -0.35,  # se mandan a gente ya desenganchada
    "newsletter": 0.05,
}

SEGMENT_EFFECT = {
    "new": 0.10,
    "engaged": 0.90,
    "casual": -0.10,
    "dormant": -0.95,
}

DEVICE_EFFECT = {
    "android": 0.00,
    "ios": 0.12,
    "web": -0.20,
}

## 3. Funciones de efecto

Modelan comportamientos conocidos del dominio:

- **Hora del día:** dos picos suaves de apertura (mañana ~8h y tarde-noche ~20h).
- **Recencia:** cuantos más días sin abrir, menor probabilidad de la próxima apertura.
- **Fatiga:** el sobre-impacto histórico de pushes resta propensión.


In [3]:
def hour_effect(hour: np.ndarray) -> np.ndarray:
    """Curva con dos picos suaves: mañana (~8h) y tarde-noche (~20h)."""
    morning = np.exp(-((hour - 8) ** 2) / 8.0)
    evening = np.exp(-((hour - 20) ** 2) / 10.0)
    return 0.6 * morning + 0.7 * evening - 0.3  # centrado alrededor de 0


def recency_effect(days: np.ndarray) -> np.ndarray:
    """Cuantos más días sin abrir, menos probable la próxima apertura."""
    return -0.45 * np.log1p(days)


def fatigue_effect(push_count: np.ndarray) -> np.ndarray:
    """Fatiga por sobre-impacto: muchos pushes históricos restan."""
    return -0.20 * np.log1p(push_count / 20.0)


def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))

## 4. Generación del dataset

Cada fila es **un evento de notificación**. Los usuarios tienen un rasgo latente de propensión a abrir (distribución Beta sesgada hacia abajo), y un mismo `user_id` aparece en varios eventos, como en un escenario real.


In [4]:
# --- Usuarios y su rasgo latente de apertura ---
user_ids = np.array([f"U{idx:06d}" for idx in range(1, N_USERS + 1)])
user_trait = rng.beta(2.0, 4.0, size=N_USERS)  # sesgada hacia abajo (~0.33)

row_user = rng.integers(0, N_USERS, size=N_ROWS)
user_id = user_ids[row_user]
trait = user_trait[row_user]

# historical_open_rate: rasgo del usuario + ruido, recortado a [0, 1]
historical_open_rate = np.clip(trait + rng.normal(0, 0.06, N_ROWS), 0.0, 1.0)

# --- Variables categóricas ---
site = rng.choice(SITES, size=N_ROWS, p=SITE_P)
campaign_type = rng.choice(CAMPAIGN_TYPES, size=N_ROWS, p=CAMPAIGN_P)
device_os = rng.choice(DEVICE_OS, size=N_ROWS, p=DEVICE_P)
segment = rng.choice(SEGMENTS, size=N_ROWS, p=SEGMENT_P)

# --- Variables temporales ---
hour_of_day = rng.integers(0, 24, size=N_ROWS)
day_of_week = rng.integers(0, 7, size=N_ROWS)  # 0 = lunes ... 6 = domingo

# --- Comportamiento histórico ---
historical_push_count = rng.poisson(40, size=N_ROWS) + 1
days_scale = np.where(trait > 0.4, 5, 20)  # los enganchados abren más reciente
days_since_last_open = rng.poisson(days_scale).astype(int)

In [5]:
# --- Logit latente: combina todos los efectos ---
weekend = np.isin(day_of_week, [5, 6]).astype(float)

z = (
    -0.55  # intercepto (calibra la tasa global de apertura)
    + 3.2 * (historical_open_rate - 0.30)
    + recency_effect(days_since_last_open)
    + hour_effect(hour_of_day)
    + fatigue_effect(historical_push_count)
    + np.vectorize(CAMPAIGN_EFFECT.get)(campaign_type)
    + np.vectorize(SEGMENT_EFFECT.get)(segment)
    + np.vectorize(DEVICE_EFFECT.get)(device_os)
    + 0.10 * weekend
    + rng.normal(0, 0.35, N_ROWS)  # ruido irreducible
)

p_open = sigmoid(z)
target_opened = rng.binomial(1, p_open)

df = pd.DataFrame({
    "user_id": user_id,
    "site": site,
    "campaign_type": campaign_type,
    "device_os": device_os,
    "hour_of_day": hour_of_day,
    "day_of_week": day_of_week,
    "historical_open_rate": np.round(historical_open_rate, 4),
    "historical_push_count": historical_push_count,
    "days_since_last_open": days_since_last_open,
    "segment": segment,
    "target_opened": target_opened.astype(int),
})

df.head(8)

,user_id,site,campaign_type,device_os,hour_of_day,day_of_week,historical_open_rate,historical_push_count,days_since_last_open,segment,target_opened
0,U004172,ecommerce,reactivation,android,5,1,0.6726,39,5,dormant,0
1,U006385,finance,promotional,android,23,3,0.3064,46,21,casual,1
2,U002329,finance,promotional,ios,10,1,0.5397,47,4,new,1
3,U007008,finance,promotional,ios,6,5,0.3155,37,18,casual,0
4,U007883,news,reactivation,android,22,5,0.1176,44,22,casual,0
5,U000922,entertainment,promotional,web,17,3,0.3008,54,26,dormant,0
6,U004091,ecommerce,promotional,ios,7,2,0.3189,42,20,casual,1
7,U000138,finance,promotional,ios,0,5,0.2822,43,25,engaged,0


## 5. Validación de la señal

Verificamos que el target se comporta de forma coherente con el dominio (no es azar): los segmentos enganchados abren más, las campañas transaccionales rinden mejor y la apertura crece monótonamente con el `historical_open_rate`.


In [6]:
print(f"Filas: {len(df):,} | Usuarios únicos: {df.user_id.nunique():,}")
print(f"Tasa de apertura global (target=1): {df.target_opened.mean():.3f}\n")

print("Apertura por segmento:")
print(df.groupby("segment")["target_opened"].mean().round(3).sort_values(ascending=False).to_string())

print("\nApertura por tipo de campaña:")
print(df.groupby("campaign_type")["target_opened"].mean().round(3).sort_values(ascending=False).to_string())

print("\nApertura por cuartil de historical_open_rate:")
q = pd.qcut(df["historical_open_rate"], 4, labels=["Q1 (bajo)", "Q2", "Q3", "Q4 (alto)"])
print(df.groupby(q, observed=True)["target_opened"].mean().round(3).to_string())

Filas: 35,000 | Usuarios únicos: 7,900
Tasa de apertura global (target=1): 0.215

Apertura por segmento:
segment
engaged    0.333
new        0.207
casual     0.179
dormant    0.091

Apertura por tipo de campaña:
campaign_type
transactional    0.305
newsletter       0.210
promotional      0.186
reactivation     0.161

Apertura por cuartil de historical_open_rate:
historical_open_rate
Q1 (bajo)    0.094
Q2           0.137
Q3           0.219
Q4 (alto)    0.408


## 6. Exportación

El CSV se guarda en `data/raw/` y se versiona con **DVC** (no entra a Git; Git solo guarda el puntero `.dvc`).


In [7]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT, index=False)

size_mb = OUTPUT.stat().st_size / 1_048_576
print(f"[OK] Dataset guardado en: {OUTPUT}")
print(f"     {len(df):,} filas · {size_mb:.2f} MB")

[OK] Dataset guardado en: ../data/raw/openrate.csv
     35,000 filas · 2.04 MB


---

## 7. Diccionario de datos — `openrate.csv`

**Caso:** predicción de *Open Rate* (apertura de notificaciones push).
**Origen:** dataset mock generado de forma reproducible (semilla = 42).
**Volumen:** 35.000 registros (eventos de notificación) · ~7.900 usuarios únicos · ~2,0 MB.
**Granularidad:** cada fila representa **un evento de notificación** enviado a un usuario.

Las variables siguen los lineamientos del **punto 9** de la actividad.

| # | Columna | Tipo | Descripción | Valores / Rango |
|---|---------|------|-------------|-----------------|
| 1 | `user_id` | texto | Identificador del usuario que recibe la notificación. Un mismo usuario puede aparecer en varios eventos. | `U000001` … `U008000` |
| 2 | `site` | categórica | Vertical o sitio que origina la notificación. | `news`, `ecommerce`, `sports`, `finance`, `entertainment` |
| 3 | `campaign_type` | categórica | Tipo de campaña de la notificación. | `promotional`, `transactional`, `reactivation`, `newsletter` |
| 4 | `device_os` | categórica | Sistema operativo / plataforma del dispositivo. | `android`, `ios`, `web` |
| 5 | `hour_of_day` | entero | Hora del día en que se envía la notificación. | `0`–`23` |
| 6 | `day_of_week` | entero | Día de la semana del envío. | `0`–`6` (0 = lunes … 6 = domingo) |
| 7 | `historical_open_rate` | flotante | Tasa histórica de apertura del usuario (propensión a abrir). Predictor de mayor peso. | `0.0`–`1.0` |
| 8 | `historical_push_count` | entero | Número de notificaciones que el usuario ha recibido históricamente. | entero positivo |
| 9 | `days_since_last_open` | entero | Días transcurridos desde la última apertura del usuario. | entero ≥ 0 |
| 10 | `segment` | categórica | Segmento de ciclo de vida del usuario. | `new`, `engaged`, `casual`, `dormant` |
| 11 | `target_opened` | binaria | **Variable objetivo.** Indica si el usuario abrió la notificación. | `0` (no abrió) · `1` (abrió) |

### Notas para el modelado (Rol 3)

- **Variable objetivo:** `target_opened`. Problema de **clasificación binaria**.
- **Balance de clases:** ~21,5 % de clase positiva. Desbalance moderado: reportar métricas más allá de *accuracy* (p. ej. *F1*, *ROC-AUC*).
- **Categóricas a codificar:** `site`, `campaign_type`, `device_os`, `segment` (*one-hot* u *ordinal encoding*).
- **`user_id`** es un identificador, **no** una variable predictora: excluirlo del entrenamiento.
- **Valores faltantes:** ninguno (dataset sintético y completo).

### Reproducibilidad

```bash
python src/generate_data.py --n-rows 35000 --seed 42 --output data/raw/openrate.csv
```

La semilla fija garantiza que el dataset sea idéntico en cada ejecución, condición necesaria para la trazabilidad del flujo MLOps.
